# Testing the Agentic Knowledge Engineering Tool
This notebook sets up the `Orchestrator` to test the Distillation, Designer, and Validation loops with `instructor` and `openai` against local Ollama models.


In [ ]:
import sys
import os
# Allow imports from local directory if running inside the ontology folder
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

from core.models import DocumentSource
from pipeline.orchestrator import Orchestrator


## Define Test Documents
Here we define the input raw texts that need to be distilled into a Knowledge Graph and synthesized into a strict Pydantic schema.


In [ ]:
doc1 = DocumentSource(
    id="doc_001",
    text_content="""
    Patient Assessment - Oct 4, 2023:
    Jordan exhibited highly repetitive physical motions, specifically hand-flapping, 
    when the classroom environment became too loud. He did not engage in verbal communication for 45 minutes.
    """
)

doc2 = DocumentSource(
    id="doc_002",
    text_content="""
    In-home Observation - Oct 5, 2023:
    During dinner, Jordan successfully maintained eye contact with his sibling when asking for water.
    However, when transitioned to bedtime, severe physical agitation and distress was observed.
    """
)

docs = [doc1, doc2]


## Initialize Orchestrator and Run Pipeline
We point the Orchestrator to the local Ollama instance (acting as an OpenAI compatible endpoint `http://localhost:11434/v1`). Note: ensure `ollama serve` is running and `mistral-small-agent` is pulled."


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

orchestrator = Orchestrator(
    model_name=os.getenv("OLLAMA_MODEL", "mistral-small-agent"), 
    base_url=os.getenv("OLLAMA_BASE_URL", "http://localhost:11434/v1"),
    hallucination_filter=True,
    ontology_depth=None,
    strict_typing=True
)

try:
    final_schema = orchestrator.run_pipeline(docs)
except Exception as e:
    print(f"Pipeline Error: {e}")
    print("Is your Ollama instance running locally, or did you export OLLAMA_BASE_URL?")
    final_schema = None


## Synthesized Schema Output
If the validation passes strictly with zero false positives, we can now view the dynamic schema produced.


In [ ]:
if final_schema:
    print("Final Dynamic Schema JSON Definition:\n")
    import json
    print(json.dumps(final_schema.model_json_schema(), indent=2))
